# Fine-tune Cellpose on Colab
Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 1. Get the project
Option A (git): set `REPO_URL` (private repo: put the token in Colab Secrets as `GH_TOKEN`).
Option B: leave `REPO_URL` empty and upload `project.zip` (containing `train_cellpose.py` and `strips/`) with the file panel.

In [ ]:
import os, subprocess
REPO_URL = ''   # e.g. 'github.com/<user>/<repo>.git'
if REPO_URL:
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN') + '@'
    except Exception:
        tok = ''
    name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
    if os.path.exists(name):
        subprocess.run(['git', '-C', name, 'pull'], check=True)
    else:
        subprocess.run(['git', 'clone', f'https://{tok}{REPO_URL}'], check=True)
    os.chdir(name)
else:
    !unzip -oq /content/project.zip -d /content/project
    os.chdir('/content/project')
print(os.getcwd()); print(os.listdir('.'))

## 2. Install

In [ ]:
!pip install -q cellpose==4.2.1.1 opencv-python-headless tifffile

## 3. Train
Uses every `strips/train/*.png` that has a corrected `_seg.npy`. Saves `models/wax_cpsam`.

In [ ]:
import glob
print(len(glob.glob('strips/train/*_seg.npy')), 'corrected train strips')
os.environ['N_EPOCHS'] = '100'
!python -u train_cellpose.py

## 4. Download the model

In [ ]:
from google.colab import files
files.download('models/wax_cpsam')